# 第12章 天文图像、FITS 与 WCS

这个 notebook 用一个极小的 cutout 表格，演示图像数组、教学版 FITS header、线性近似 WCS 和最小孔径测光工作流。

## 学习目标

- 把像素矩阵重建出来
- 读取一个最小教学版 header / WCS 描述
- 做背景估计和孔径求和
- 把中心像素近似转换到天空坐标
- 理解图像数据如何走向物理解释

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from pathlib import Path

DATA_PATH = Path('../../data/small/fits_wcs_cutout_demo.csv')

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

pixels = [
    {
        'x_pix': int(row['x_pix']),
        'y_pix': int(row['y_pix']),
        'counts': float(row['counts']),
    }
    for row in rows
]

x_max = max(row['x_pix'] for row in pixels)
y_max = max(row['y_pix'] for row in pixels)
image = [[0.0 for _ in range(x_max + 1)] for _ in range(y_max + 1)]
for row in pixels:
    image[row['y_pix']][row['x_pix']] = row['counts']

print(f'Loaded {len(pixels)} pixels from {DATA_PATH.name}')
print('Image shape:', (y_max + 1, x_max + 1))
print('Python version:', platform.python_version())
print('Central row:', image[3])


In [ ]:
header = {
    'CRPIX1': 4.0,
    'CRPIX2': 4.0,
    'CRVAL1': 150.1230,
    'CRVAL2': 2.3450,
    'CDELT1': -0.0002777778,
    'CDELT2': 0.0002777778,
    'EXPTIME': 120.0,
    'FILTER': 'G',
}

def pixel_to_sky(x_pix, y_pix, hdr):
    dx = (x_pix + 1) - hdr['CRPIX1']
    dy = (y_pix + 1) - hdr['CRPIX2']
    dec = hdr['CRVAL2'] + dy * hdr['CDELT2']
    ra = hdr['CRVAL1'] + dx * hdr['CDELT1'] / math.cos(math.radians(hdr['CRVAL2']))
    return ra, dec

source_ra_deg, source_dec_deg = pixel_to_sky(3, 3, header)
print('Teaching header keys:', sorted(header))
print('Central source sky position [deg]:', round(source_ra_deg, 6), round(source_dec_deg, 6))


In [ ]:
background_pixels = [
    row['counts']
    for row in pixels
    if row['x_pix'] in {0, x_max} or row['y_pix'] in {0, y_max}
]
background_mean = sum(background_pixels) / len(background_pixels)

aperture_pixels = []
for row in pixels:
    dx = row['x_pix'] - 3
    dy = row['y_pix'] - 3
    radius = math.sqrt(dx * dx + dy * dy)
    if radius <= 1.5:
        aperture_pixels.append(row['counts'])

aperture_sum = sum(aperture_pixels)
n_pix = len(aperture_pixels)
net_flux = aperture_sum - n_pix * background_mean
instrumental_mag = -2.5 * math.log10(net_flux)

print('Background mean:', round(background_mean, 3))
print('Aperture pixel count:', n_pix)
print('Aperture sum:', round(aperture_sum, 3))
print('Net flux:', round(net_flux, 3))
print('Instrumental magnitude:', round(instrumental_mag, 3))


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print('matplotlib unavailable; skipped image display:', exc)
else:
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    ax.imshow(image, origin='lower', cmap='magma')
    ax.scatter([3], [3], facecolors='none', edgecolors='cyan', s=120)
    ax.set_title('Teaching FITS/WCS Cutout')
    ax.set_xlabel('x pixel')
    ax.set_ylabel('y pixel')
    fig.tight_layout()
    print('Displayed cutout image with source marker.')


## 解释

这个最小示例把图像处理拆成了清楚的链条：先把 cutout 重建成二维数组，再用教学版 header 给出 WCS 参考，随后做背景估计和孔径求和，最后把像素位置转换到天空坐标。真正重要的不是这一小块数据有多真实，而是你已经开始用图像数组、元数据和物理解释去看一张天文图像。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_pixels': len(pixels),
    'shape': (y_max + 1, x_max + 1),
    'background_mean': round(background_mean, 3),
    'aperture_n_pix': n_pix,
    'net_flux': round(net_flux, 3),
    'instrumental_mag': round(instrumental_mag, 3),
    'source_ra_deg': round(source_ra_deg, 6),
    'source_dec_deg': round(source_dec_deg, 6),
    'python_version': platform.python_version(),
}

print('FITS/WCS snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')


## V2 Evidence Record artifact

本章的记录重点是图像数组、header/WCS、背景估计和孔径设置，而不是只保存一张显示图。


In [ ]:
artifact_dir = Path('results')
artifact_dir.mkdir(parents=True, exist_ok=True)
evidence_path = artifact_dir / 'ch12_evidence_record.md'

edge_pixel_count = len(background_pixels)
image_shape = (y_max + 1, x_max + 1)
pixel_scale_arcsec = abs(header['CDELT2']) * 3600.0
possible_ml_shape = f'{image_shape[0]} x {image_shape[1]} x 1'

record_text = f"""# Evidence Record - Ch12 FITS Images and WCS

## Record Type

- Record type: figure / preprocessing / data check

## 1. Input

- Data / file path, preferably relative to project root: `{DATA_PATH}`
- Data source or generation method: AIforAstronomers teaching cutout table standing in for a FITS image
- Script / notebook: `ch12_fits_images_wcs.ipynb`
- Code version / tag, if relevant: fill in the current Git commit or release tag when submitting

## 2. Operation

- Command or entry point: run notebook cells in order
- Key parameters: center pixel `(3, 3)`; aperture radius `1.5` pixels; edge pixels used for background
- Random seed, if relevant: not applicable
- Output directory: `results/`

## 3. Output

- Output file(s): `{evidence_path}`; optional display plot if matplotlib is available
- Short result summary: image shape `{image_shape}`; background mean `{background_mean:.3f}`; net flux `{net_flux:.3f}`; instrumental magnitude `{instrumental_mag:.3f}`
- Generated by which script/notebook: `ch12_fits_images_wcs.ipynb`

## 4. Check

- Check performed: reconstructed pixel grid shape, inspected teaching header keys, converted source pixel to sky position, checked background and aperture counts
- Evidence from the check: `{len(pixels)}` pixels loaded; `{edge_pixel_count}` background pixels; `{n_pix}` aperture pixels; source sky position `({source_ra_deg:.6f}, {source_dec_deg:.6f})` deg

## 5. Limit

- Known limitation: CSV cutout and simple header are teaching substitutes, not a full FITS HDU with real WCS distortion or calibration metadata
- Selection / quality / uncertainty issue, if relevant: no bad-pixel mask, variance image, saturation flag, or calibrated photometric zero point

## 6. Reuse in ML

- Sample / feature / label: one small image cutout could become a model sample; counts array could become image features
- Uncertainty / quality flag: no formal uncertainty map or mask in this teaching table
- Preprocessing record: records cutout shape, background method, aperture choice, and approximate WCS
- Reproducibility evidence: source path, header values, aperture settings, and checks are recorded

## Ch12 Fields

- FITS file: teaching substitute `{DATA_PATH}`
- HDU/header checked: header keys `{sorted(header)}`
- image shape: `{image_shape}`
- pixel scale / WCS: approximately `{pixel_scale_arcsec:.3f}` arcsec/pixel; CRVAL `({header['CRVAL1']}, {header['CRVAL2']})`
- background method: mean of edge pixels
- aperture or cutout setting: radius `1.5` pixels around `(3, 3)`; cutout shape `{image_shape}`
- normalization: raw counts retained; no normalization applied
- mask / bad pixels: no mask or bad-pixel flags in this teaching table
- possible ML input shape: `{possible_ml_shape}`
"""

evidence_path.write_text(record_text, encoding='utf-8')
print('wrote Evidence Record:', evidence_path)
